In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import pandas as pd
import numpy as np
import os

2026-04-19 12:04:48.353413: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776600288.629181      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776600288.712250      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776600289.398579      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776600289.398622      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776600289.398627      55 computation_placer.cc:177] computation placer alr

In [7]:
train_dir = '/kaggle/input/competitions/cs-460-muffin-vs-chihuahua-classification-challenge/train'                     # training folder
test_dir  = '/kaggle/input/competitions/cs-460-muffin-vs-chihuahua-classification-challenge/kaggle_test_final'        # test folder
img_size  = (128, 128)
batch_size = 32
epochs = 20

In [8]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,

    rotation_range=25,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.25,
    shear_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.7,1.3]
)

val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

In [9]:
train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='training'
)

val_data = val_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='validation'
)

Found 3788 images belonging to 2 classes.
Found 945 images belonging to 2 classes.


In [10]:
print("Class mapping:", train_data.class_indices)

Class mapping: {'chihuahua': 0, 'muffin': 1}


In [11]:
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(128, activation='relu',
                 kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.5),

    layers.Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-04-19 12:06:42.944538: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [12]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [13]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

lr_reduce = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=2
)

In [14]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=epochs,
    callbacks=[early_stop, lr_reduce]
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 104s 851ms/step - accuracy: 0.6352 - loss: 0.7853 - val_accuracy: 0.6360 - val_loss: 0.7492 - learning_rate: 0.0010
Epoch 2/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 87s 730ms/step - accuracy: 0.7681 - loss: 0.5284 - val_accuracy: 0.8667 - val_loss: 0.4004 - learning_rate: 0.0010
Epoch 3/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 87s 726ms/step - accuracy: 0.8117 - loss: 0.4634 - val_accuracy: 0.8857 - val_loss: 0.3249 - learning_rate: 0.0010
Epoch 4/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 86s 725ms/step - accuracy: 0.8058 - loss: 0.4572 - val_accuracy: 0.8878 - val_loss: 0.3231 - learning_rate: 0.0010
Epoch 5/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 86s 724ms/step - accuracy: 0.8270 - loss: 0.4377 - val_accuracy: 0.8910 - val_loss: 0.2996 - learning_rate: 0.0010
Epoch 6/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 87s 727ms/step - accuracy: 0.8440 - loss: 0.4165 - val_accuracy: 0.9079 - val_loss: 0.2786 - learning_rate: 0.0010
Epoch 7/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 87s 725ms/step - accuracy: 0.

In [16]:
files = os.listdir(test_dir)

if len(files) > 0 and files[0].endswith(('.jpg','.png','.jpeg')):
    print("Using flow_from_dataframe (flat folder detected)")

    df = pd.DataFrame({'filename': files})

    test_datagen = ImageDataGenerator(rescale=1./255)

    test_data = test_datagen.flow_from_dataframe(
        dataframe=df,
        directory=test_dir,
        x_col='filename',
        y_col=None,
        target_size=img_size,
        batch_size=1,
        class_mode=None,
        shuffle=False
    )

    filenames = df['filename']

else:
    print("Using flow_from_directory (subfolder structure detected)")

    test_datagen = ImageDataGenerator(rescale=1./255)

    test_data = test_datagen.flow_from_directory(
        test_dir,
        target_size=img_size,
        batch_size=1,
        class_mode=None,
        shuffle=False
    )

    filenames = [f.split('/')[-1] for f in test_data.filenames]

Using flow_from_dataframe (flat folder detected)
Found 1138 validated image filenames.


In [17]:
preds = model.predict(test_data)

# Adjust based on class mapping
class_map = train_data.class_indices
inv_map = {v:k for k,v in class_map.items()}

labels = []
for p in preds:
    label = inv_map[1] if p > 0.5 else inv_map[0]
    labels.append(label)


1138/1138 ━━━━━━━━━━━━━━━━━━━━ 14s 12ms/step


In [18]:
submission = pd.DataFrame({
    'ID': filenames,
    'Predict': labels
})

submission.to_csv('submission1.csv', index=False)

print("✅ submission.csv created!")

✅ submission.csv created!
